# 05 - Le simulateur go / no-go

`src/simulator.py` : parmi les jeux comparables, la **probabilité de rentabilité** est la part de ceux dont le revenu net dépasse le budget du studio.
Seuils : GO à partir de 50 %, RISKY de 25 à 50 %, NO-GO en dessous.

In [1]:
import sys
sys.path.append('..')

import pandas as pd
from src.cleaning import load_clean
from src.estimation import add_estimates
from src.simulator import GameConcept, break_even_probability, recommendation

df = add_estimates(load_clean('../data/games_clean.csv'))

## 1. Tester les fonctions de base

In [2]:
for p in [0.8, 0.5, 0.3, 0.25, 0.1]:
    print(p, '->', recommendation(p))

0.8 -> GO
0.5 -> GO
0.3 -> RISKY
0.25 -> RISKY
0.1 -> NO-GO


## 2. Le check du guide : Indie + Roguelike, 15 $, 50 000 $ puis 500 000 $

In [3]:
result = GameConcept(['Indie', 'Roguelike'], 15, 50000).evaluate(df)
result

{'tags': ['Indie', 'Roguelike'],
 'price': 15.0,
 'budget': 50000.0,
 'n_comparables': 625,
 'revenue_range': (2704.59, 16782.444, 100447.45199999999),
 'probabilities': {'low': 0.2544, 'mid': 0.344, 'high': 0.4016},
 'recommendation': 'RISKY',
 'recommendations': {'low': 'RISKY', 'mid': 'RISKY', 'high': 'RISKY'},
 'recommended_price': '20-30$',
 'warning': None}

In [4]:
for budget in [50_000, 500_000]:
    concept = GameConcept(['Indie', 'Roguelike'], 15, budget)
    concept.evaluate(df)
    print(concept.summary())

625 jeux comparables, 34% ont remboursé 50 000 $ : RISKY. Prix conseillé : 20-30$.


625 jeux comparables, 10% ont remboursé 500 000 $ : NO-GO. Prix conseillé : 20-30$.


Le budget passe de 50 000 $ à 500 000 $ : la probabilité tombe de 34 % à 10 %, le verdict passe de RISKY à NO-GO. Logique.

## 3. Notre fil rouge : 2 personnes, 60 000 $, roguelike ou cozy game à 15 $ ?

In [5]:
rows = []
for name, tags in {'A - Roguelike': ['Indie', 'Roguelike'], 'B - Cozy': ['Indie', 'Cozy']}.items():
    concept = GameConcept(tags, 15, 60000)
    r = concept.evaluate(df)
    print(name, ':', concept.summary())
    for scenario in ['low', 'mid', 'high']:
        rows.append({'concept': name, 'scénario': scenario,
                     'probabilité': round(r['probabilities'][scenario], 2),
                     'verdict': r['recommendations'][scenario]})
pd.DataFrame(rows).pivot(index='concept', columns='scénario', values=['probabilité', 'verdict'])

A - Roguelike : 625 jeux comparables, 32% ont remboursé 60 000 $ : RISKY. Prix conseillé : 20-30$.


B - Cozy : 119 jeux comparables, 40% ont remboursé 60 000 $ : RISKY. Prix conseillé : 15-20$.


probabilité             verdict              
scénario             high   low   mid    high    low    mid
concept                                                    
A - Roguelike        0.37  0.24  0.32   RISKY  NO-GO  RISKY
B - Cozy             0.45  0.29   0.4   RISKY  RISKY  RISKY

**Réponse au fil rouge** : les deux concepts sont **RISKY** à 60 000 $, mais le **cozy game** a plus de chances (40 % contre 32 %) et une fourchette de revenu plus haute.
**Sensibilité (étape 6)** : le cozy game reste RISKY dans les 3 scénarios, alors que le roguelike **bascule en NO-GO** dans le scénario bas (24 %, juste sous le seuil de 25 %). Le concept cozy est donc le plus robuste aux erreurs d'estimation.
Attention : la niche cozy a peu de comparables (environ 120) et ses sorties ont triplé depuis 2021-22.

## 4. Trois concepts contrastés

In [6]:
tests = {
    'Petit budget, niche porteuse (Open World Survival Craft, 15 $, 20 000 $)': (['Open World Survival Craft'], 15, 20000),
    'Gros budget, niche saturée (Clicker, 5 $, 300 000 $)': (['Clicker'], 5, 300000),
    'Niche presque vide (Roguelike Deckbuilder + Trains, 15 $, 60 000 $)': (['Roguelike Deckbuilder', 'Trains'], 15, 60000),
}
for name, args in tests.items():
    concept = GameConcept(*args)
    concept.evaluate(df)
    print(name, '\n   ', concept.summary())

Petit budget, niche porteuse (Open World Survival Craft, 15 $, 20 000 $) 
    216 jeux comparables, 54% ont remboursé 20 000 $ : GO. Prix conseillé : 30$+.


Gros budget, niche saturée (Clicker, 5 $, 300 000 $) 
    724 jeux comparables, 1% ont remboursé 300 000 $ : NO-GO. Prix conseillé : 30$+.


Niche presque vide (Roguelike Deckbuilder + Trains, 15 $, 60 000 $) 
    1 jeux comparables, 100% ont remboursé 60 000 $ : RISKY. Prix conseillé : non déterminé. Attention : Seulement 1 jeux comparables (minimum conseillé : 30) : résultat peu fiable.


Les résultats ont du sens : GO pour la niche porteuse à petit budget, NO-GO pour le clicker à gros budget, et un avertissement (verdict plafonné à RISKY) quand il n'y a qu'un comparable.

## 5. Cas limites : aucun plantage

In [7]:
for args in [([], 15, 60000), (['Indie'], -5, 60000), (['Indie'], 15, 0)]:
    try:
        GameConcept(*args)
    except ValueError as error:
        print('Entrée refusée proprement :', error)

empty = GameConcept(['Tag qui n existe pas'], 15, 60000)
empty.evaluate(df)
print(empty.summary())

Entrée refusée proprement : Choisissez au moins un tag.
Entrée refusée proprement : Le prix doit être strictement positif.
Entrée refusée proprement : Le budget doit être strictement positif.


0 jeux comparables, 0% ont remboursé 60 000 $ : NO-GO. Prix conseillé : non déterminé. Attention : Aucun jeu comparable : impossible d'évaluer ce concept.
